# V3 — Corner Detection + Corner Classification : rapport final

Étude complète de l'approche "coin". Voir `reports/v3_final_report.md` pour la synthèse écrite.

**TL;DR** : détecteur de coins excellent (F1 réel 0.985), mais le corner-classifier souffre d'un gap synthétique→réel (0.999 train → 0.759 réel) qui, combiné à la multiplication d'erreurs du pipeline multi-étapes, plafonne le end-to-end à 0.41 vs V2 0.787. Submission = V2.

## 1. Architecture & compte de paramètres

In [ ]:
import sys, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print(subprocess.run([sys.executable, str(ROOT/'scripts'/'count_params.py')],
                     capture_output=True, text=True).stdout)

## 2. Dataset synthétique
5000 scènes, 67 731 coins. QA visuel :

In [ ]:
import cv2, matplotlib.pyplot as plt
m = cv2.imread(str(ROOT/'reports'/'v3_synthetic_qa'/'mosaic.jpg'))
if m is not None:
    plt.figure(figsize=(20,9)); plt.imshow(cv2.cvtColor(m, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 3. Résultats par composant

| Composant | Métrique | Valeur |
|---|---|---|
| Corner detector (Phase A synth) | P/R | 0.999 / 0.999 |
| Corner detector (Phase B réel) | F1 réel | **0.985** |
| Corner classifier | acc train | 0.999 |
| Corner classifier | **acc réel** | **0.759** |

Courbes d'entraînement :

In [ ]:
for log in ['corner_det_phaseA.log', 'corner_det_phaseB.log', 'corner_clf.log']:
    p = ROOT/'logs'/log
    if p.exists():
        lines = p.read_text().strip().split('\n')
        print(f'--- {log} (head/tail) ---')
        print('\n'.join(lines[:2] + ['...'] + lines[-3:]))
        print()

## 4. Résultat end-to-end & comparaison V2

| Variante | CenterAcc | ActiveAcc | F1 | Score |
|---|---|---|---|---|
| V3 (3 variantes dédup) | 0.07-0.10 | 0.741 | 0.38-0.41 | **0.39-0.41** |
| **V2 (submission)** | 0.85+ | 0.80 | 0.79 | **0.787** |

In [ ]:
import pandas as pd
d = pd.read_csv(ROOT/'reports'/'v3_eval.csv')
print('V3 par-image (extrait) :')
print(d[['image_id','n_gt','n_pred','f1','center_ok','active_ok']].head(10).to_string(index=False))
print(f"\nCenterAcc={d.center_ok.mean():.3f} ActiveAcc={d.active_ok.mean():.3f} F1={d.f1.mean():.3f}")

## 5. Failure analysis — pourquoi V3 plafonne

1. **Gap synthétique→réel** du corner-classifier : 0.999 → 0.759. Crops de coins synthétiques ≠ vrais coins (usure/reflets/flou).
2. **Multiplication d'erreurs** : 0.99 (det) × 0.76 (clf/coin) × accord 2 coins ≪ classifieur full-card V2 (contexte complet en 1 passe).
3. **Dilemme pairing** : accord-label → cartes 1-coin mal localisées (CenterAcc 0.07) ; géométrique pur → mé-pairing voisins (F1 0.38).

## 6. Leçon

Un sous-composant excellent (détecteur 0.99) ne sauve pas un pipeline si la **composition des incertitudes** domine. Moins de maillons = moins de multiplication d'erreurs → le classifieur full-card V2 (1 passe, contexte complet) reste supérieur ici.

## 7. Décision : **submission = V2 (0.787)**. V3 conservé comme étude (détecteur de coins 0.985 réutilisable, dataset synthétique, analyse quantifiée).